In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Загрузка
train = pd.read_csv("../data/raw/UNSW_NB15_training-set.csv")
test  = pd.read_csv("../data/raw/UNSW_NB15_testing-set.csv")

y_train, y_test = train["label"], test["label"]
drop_cols = ["id", "attack_cat", "label"]
X_train = train.drop(columns=drop_cols)
X_test  = test.drop(columns=drop_cols)

# 2. log1p перекошенных признаков
skewed = ["dur", "spkts", "dpkts", "sbytes", "dbytes", "rate",
          "sload", "dload", "sloss", "dloss", "sinpkt", "dinpkt",
          "sjit", "djit", "response_body_len"]
for df in (X_train, X_test):
    df[skewed] = np.log1p(df[skewed])

# 3. Масштабирование + One-Hot
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
print("Числовые:", len(num_cols), "| Категориальные:", cat_cols)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared  = preprocessor.transform(X_test)

# 4. Сохранение
os.makedirs("../data/processed", exist_ok=True)
np.savez_compressed("../data/processed/prepared.npz",
                    X_train=X_train_prepared, X_test=X_test_prepared,
                    y_train=y_train.values, y_test=y_test.values)
joblib.dump(preprocessor, "../models/preprocessor.joblib")

print("Готово:", X_train_prepared.shape, X_test_prepared.shape)

C:\Users\User\AppData\Local\Temp\ipykernel_31660\1118928748.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include="object").columns.tolist()


Числовые: 39 | Категориальные: ['proto', 'service', 'state']
Готово: (175341, 194) (82332, 194)
